In [4]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [7]:
# Simulation of the protocol with an attacker being detected.
simulator = BasicSimulator()


# Quantum Random Number Generator
def quantum_random_bits(n):
    bits  = []
    batch = 20
    while len(bits) < n:
        size = min(batch, n - len(bits))
        qc   = QuantumCircuit(size, size)
        qc.h(range(size))
        qc.measure(range(size), range(size))
        job    = simulator.run(transpile(qc, simulator), shots=1)
        result = list(job.result().get_counts().keys())[0]
        bits  += [int(b) for b in reversed(result)]
    return bits[:n]


# Alice prepares and sends qubits
def alice_prepare(n):
    bits  = quantum_random_bits(n)
    bases = quantum_random_bits(n)

    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)       # encode bit value
        if basis == 1:
            qc.h(0)       # rotate to diagonal basis
        circuits.append(qc)

    print("[Alice] bits:  ", bits)
    print("[Alice] bases: ", bases, "  (0=rectilinear, 1=diagonal)")
    return circuits, bits, bases


# Eve intercepts, measures, re-sends (intercept-and-resend)
def eve_intercept(circuits):
    eve_bases   = quantum_random_bits(len(circuits))
    eve_results = []
    forwarded   = []          # circuits Eve sends on to Bob

    for qc, basis in zip(circuits, eve_bases):
        eve_qc = QuantumCircuit(1, 1)
        eve_qc.compose(qc, inplace=True)   # apply Alice's encoding
        if basis == 1:
            eve_qc.h(0)                    # rotate to Eve's basis
        eve_qc.measure(0, 0)

        job = simulator.run(transpile(eve_qc, simulator), shots=1)
        measured_bit = int(list(job.result().get_counts().keys())[0])
        eve_results.append(measured_bit)

        new_qc = QuantumCircuit(1, 1)
        if measured_bit == 1:
            new_qc.x(0)
        if basis == 1:
            new_qc.h(0)
        forwarded.append(new_qc)

    print("[Eve]   bases:   ", eve_bases)
    print("[Eve]   results: ", eve_results)
    print(f"[Eve]   intercepted all {len(circuits)} qubits — Bob receives Eve's re-sent qubits")
    return forwarded, eve_bases, eve_results


# Bob receives and measures qubits from Eve
def bob_measure(circuits, n):
    bases   = quantum_random_bits(n)
    results = []

    for qc, basis in zip(circuits, bases):
        full_qc = QuantumCircuit(1, 1)
        full_qc.compose(qc, inplace=True)
        if basis == 1:
            full_qc.h(0)      # rotate from diagonal basis before measuring
        full_qc.measure(0, 0)

        job = simulator.run(transpile(full_qc, simulator), shots=1)
        results.append(int(list(job.result().get_counts().keys())[0]))

    print("[Bob]   bases:   ", bases)
    print("[Bob]   results: ", results)
    return results, bases


# Sifting over classical public channel (Alice compared with Bob)
def sift_key(alice_bits, alice_bases, bob_results, bob_bases):
    alice_key, bob_key, kept = [], [], []

    for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
        if ab == bb:
            alice_key.append(alice_bits[i])
            bob_key.append(bob_results[i])
            kept.append(i)

    pct = len(alice_key) / len(alice_bits) * 100
    print(f"\n[Sift]  Kept {len(alice_key)}/{len(alice_bits)} bits ({pct:.0f}%)")
    print("[Alice sifted key]", alice_key)
    print("[Bob   sifted key]", bob_key)
    return alice_key, bob_key


# Error rate check — attack detection
def check_error_rate(alice_key, bob_key, sample_fraction=0.2, threshold=0.15):
    n_sample = max(1, math.ceil(len(alice_key) * sample_fraction))

    # Use quantum RNG to pick which positions to sample
    rand_bits = quantum_random_bits(len(alice_key))
    indices   = [i for i, b in enumerate(rand_bits) if b == 1][:n_sample]
    if not indices:
        indices = [0]

    errors = sum(alice_key[i] != bob_key[i] for i in indices)
    qber   = errors / len(indices)

    print(f"\n[Check] Sample size: {len(indices)}, Errors: {errors}")
    print(f"[Check] QBER: {qber:.2%}  (security threshold: {threshold:.0%})")

    if qber > threshold:
        print("[Check] QBER exceeds threshold — Eve detected! Key discarded.")
        return qber, True
    else:
        print("[Check] QBER within safe range — attack not detected this run.")
        return qber, False


N = 100   # number of qubits Alice sends

print("=" * 56)
print("BB84 — With attacker (Eve: intercept-and-resend)")
print("=" * 56)

#1. Alice encodes qubits
circuits, alice_bits, alice_bases = alice_prepare(N)

#2. Eve intercepts every qubit, measures, and re-sends
print()
forwarded, eve_bases, eve_results = eve_intercept(circuits)

#3. Bob measures Eve's re-sent qubits
print()
bob_results, bob_bases = bob_measure(forwarded, N)

#4. Alice and Bob sift on the classical channel
alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_results, bob_bases)

#5. Check error rate and decide whether to trust the key
qber, attack_detected = check_error_rate(alice_key, bob_key)


print("\n── Summary " + "─" * 45)
print(f"  Qubits sent:         {N}")
print(f"  Sifted key bits:     {len(alice_key)}")
print(f"  QBER:                {qber:.2%}")
print(f"  Attack detected:     {'YES' if attack_detected else 'NO  (try larger N for reliability)'}")
if attack_detected:
    print("\n  Protocol aborted — shared key is insecure and discarded.")
else:
    print("\n  Key accepted (Eve got lucky this run — increase N to reduce this risk).")

BB84 — With attacker (Eve: intercept-and-resend)
[Alice] bits:   [1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1]
[Alice] bases:  [0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1]   (0=rectilinear, 1=diagonal)

[Eve]   bases:    [1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 